# Google Shopping – 3 ofertas mais baratas (template DDL)

Lê os itens **selecionados** da aba **Entrega**, busca cada um no **Google Shopping** e grava as **3 ofertas mais baratas** na aba **Google Shopping** da mesma planilha.

### Como usar
1. Na célula **⚙️ Configuração**, cole o link da planilha.
2. Menu **Ambiente de execução → Executar tudo**.
3. Autorize sua conta Google quando o Colab pedir.

### Regras
- Entra na busca a linha com **`Selecionado?`** preenchido e diferente de `No`.
- Variações do mesmo produto (mesmo **Item ID**) são buscadas uma vez só.
- A pesquisa usa o **Item Name**.
- A aba **Entrega** não é alterada.

> ⚠️ É raspagem direta: o Google pode pedir **captcha**. Se isso acontecer, a busca para e os itens restantes ficam `PENDENTE`. Espere alguns minutos e rode só a última célula de novo: os itens já `OK` são mantidos.

In [ ]:
# ⚙️ CONFIGURAÇÃO — único lugar que precisa ser alterado
PLANILHA_URL = "https://docs.google.com/spreadsheets/d/COLE_O_ID_AQUI/edit"

QTD_OFERTAS            = 3       # quantas ofertas mais baratas trazer por item
FILTRAR_TITULO         = True    # descarta anúncios cujo título não tem a maioria das palavras do item
PULAR_JA_CONCLUIDOS    = True    # não refaz itens que já estão OK na aba de resultado
PAUSA_ENTRE_BUSCAS_SEG = (4, 9)  # pausa aleatória (mín, máx) entre buscas – reduz risco de captcha

## 1. Instalação *(só na primeira execução da sessão)*

In [ ]:
!pip install -q playwright gspread
!playwright install --with-deps chromium > /dev/null 2>&1
print("Instalação concluída")

## 2. Estrutura do template
Nomes de abas e colunas. Só mude se o template mudar.

In [ ]:
# Estrutura fixa do template. Só mude se o template mudar.
ABA_ORIGEM      = "Entrega"
LINHA_CABECALHO = 4                  # linha (1, 2, 3...) onde ficam os nomes das colunas
ABA_DESTINO     = "Google Shopping"  # criada automaticamente se não existir


class ColunaOrigem:
    """Cabeçalhos lidos da aba Entrega (localizados pelo nome, não pela letra)."""
    SELECIONADO = "Selecionado?"
    ITEM_ID     = "Item ID"
    ITEM_NAME   = "Item Name"
    SHOP_NAME   = "Shop Name"
    FINAL_PRICE = "Final Price"

    @classmethod
    def todas(cls):
        return [cls.SELECIONADO, cls.ITEM_ID, cls.ITEM_NAME, cls.SHOP_NAME, cls.FINAL_PRICE]


VALOR_NAO_SELECIONADO = "No"

COLUNAS_DESTINO = [
    "Item ID", "Item Name", "Shop Name", "Final Price (Shopee)",
    "Posição", "Preço Google (R$)", "Loja", "Título do anúncio", "Link",
    "Status", "Atualizado em",
]


class Status:
    OK            = "OK"
    SEM_RESULTADO = "SEM RESULTADO"
    BLOQUEADO     = "BLOQUEADO (captcha)"
    PENDENTE      = "PENDENTE"   # não buscado porque o Google bloqueou antes
    ERRO          = "ERRO"

## 3. Modelos de dados

In [ ]:
from dataclasses import dataclass, field


@dataclass(frozen=True)
class Item:
    """Produto da aba Entrega que será pesquisado."""
    item_id: str
    nome: str
    loja_shopee: str
    preco_shopee: object  # número quando possível


@dataclass(frozen=True)
class Oferta:
    """Uma oferta encontrada no Google Shopping."""
    titulo: str
    preco: float
    loja: str
    link: str


@dataclass
class ResultadoBusca:
    item: Item
    status: str
    ofertas: list = field(default_factory=list)

## 4. Leitura e escrita na planilha

In [ ]:
from datetime import datetime

import gspread


def abrir_planilha(cliente, url_ou_id):
    if url_ou_id.startswith("http"):
        return cliente.open_by_url(url_ou_id)
    return cliente.open_by_key(url_ou_id)


def localizar_colunas(cabecalho, nomes):
    """Devolve {nome: índice} e avisa claramente se alguma coluna sumiu do template."""
    cabecalho_limpo = [c.strip() for c in cabecalho]
    faltando = [n for n in nomes if n not in cabecalho_limpo]
    if faltando:
        raise ValueError(f"Coluna(s) não encontrada(s) na linha {LINHA_CABECALHO} da aba "
                         f"'{ABA_ORIGEM}': {faltando}")
    return {n: cabecalho_limpo.index(n) for n in nomes}


def texto_para_numero(texto):
    """'1,234.56' -> 1234.56. Se não for número, devolve o texto original."""
    try:
        return float(texto.replace(",", ""))
    except ValueError:
        return texto.strip()


def esta_selecionado(valor):
    valor = valor.strip()
    return bool(valor) and valor != VALOR_NAO_SELECIONADO


def ler_itens_selecionados(planilha):
    """Itens com 'Selecionado?' preenchido e diferente de 'No', sem repetir Item ID."""
    linhas = planilha.worksheet(ABA_ORIGEM).get_values()
    cabecalho, dados = linhas[LINHA_CABECALHO - 1], linhas[LINHA_CABECALHO:]
    col = localizar_colunas(cabecalho, ColunaOrigem.todas())

    itens = {}
    for linha in dados:
        linha = linha + [""] * (len(cabecalho) - len(linha))
        item_id = linha[col[ColunaOrigem.ITEM_ID]].strip()
        if item_id and item_id not in itens and esta_selecionado(linha[col[ColunaOrigem.SELECIONADO]]):
            itens[item_id] = Item(
                item_id=item_id,
                nome=linha[col[ColunaOrigem.ITEM_NAME]].strip(),
                loja_shopee=linha[col[ColunaOrigem.SHOP_NAME]].strip(),
                preco_shopee=texto_para_numero(linha[col[ColunaOrigem.FINAL_PRICE]]),
            )
    return list(itens.values())


def obter_aba_destino(planilha):
    try:
        return planilha.worksheet(ABA_DESTINO)
    except gspread.WorksheetNotFound:
        return planilha.add_worksheet(ABA_DESTINO, rows=1000, cols=len(COLUNAS_DESTINO))


def ler_linhas_concluidas(aba_destino):
    """Linhas de execuções anteriores cujo item terminou com status OK."""
    linhas = aba_destino.get_values(value_render_option="UNFORMATTED_VALUE")[1:]
    i_status = COLUNAS_DESTINO.index("Status")
    return [l for l in linhas if len(l) > i_status and l[i_status] == Status.OK]


def resultado_para_linhas(resultado):
    """Uma linha por oferta; se não houver oferta, uma linha só com o status."""
    item = resultado.item
    base = [item.item_id, item.nome, item.loja_shopee, item.preco_shopee]
    agora = datetime.now().strftime("%d/%m/%Y %H:%M")
    if not resultado.ofertas:
        return [base + ["", "", "", "", "", resultado.status, agora]]
    return [base + [pos, o.preco, o.loja, o.titulo, o.link, resultado.status, agora]
            for pos, o in enumerate(resultado.ofertas, start=1)]


def gravar_resultados(aba_destino, linhas):
    aba_destino.clear()
    # RAW mantém o Item ID como texto (sem virar notação científica)
    aba_destino.update([COLUNAS_DESTINO] + linhas, value_input_option="RAW")
    aba_destino.format("A1:K1", {"textFormat": {"bold": True}})
    aba_destino.freeze(rows=1)

## 5. Busca no Google Shopping
A extração não depende das classes CSS do Google (que mudam sempre). Ela localiza os textos com `R$`, sobe até o card do produto e lê título, loja e link.

In [ ]:
import re
import unicodedata
from urllib.parse import quote_plus, urlparse, parse_qs

URL_BUSCA = "https://www.google.com/search?tbm=shop&hl=pt-BR&gl=br&tbs=p_ord:p&q={termo}"
USER_AGENT = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
              "(KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36")
TIMEOUT_CARREGAMENTO_MS = 45_000
ESPERA_RENDERIZACAO_MS  = 2_500
ESPERA_ROLAGEM_MS       = 1_500
SINAIS_DE_CAPTCHA = ("unusual traffic", "tráfego incomum", "recaptcha")
BOTOES_COOKIES = ("Aceitar tudo", "Accept all")
PROPORCAO_MINIMA_PALAVRAS = 0.6


class BloqueioDoGoogle(Exception):
    """O Google exibiu captcha / página de tráfego incomum."""


# ---------- Regras de texto (puras, fáceis de testar) ----------

def extrair_preco(texto):
    """'R$ 1.234,56' -> 1234.56; None se não houver preço."""
    achado = re.search(r"R\$\s*(\d{1,3}(?:\.\d{3})*|\d+)(?:,(\d{2}))?", texto or "")
    if not achado:
        return None
    reais, centavos = achado.group(1).replace(".", ""), achado.group(2) or "00"
    return float(f"{reais}.{centavos}")


def _sem_acentos(texto):
    decomposto = unicodedata.normalize("NFKD", texto.lower())
    return "".join(c for c in decomposto if not unicodedata.combining(c))


def titulo_corresponde_ao_item(nome_item, titulo):
    """True se o título contém a maioria das palavras relevantes do item."""
    palavras = [p for p in re.findall(r"\w+", _sem_acentos(nome_item)) if len(p) > 2]
    if not palavras:
        return True
    titulo_normalizado = _sem_acentos(titulo)
    encontradas = sum(p in titulo_normalizado for p in palavras)
    return encontradas / len(palavras) >= PROPORCAO_MINIMA_PALAVRAS


def link_final(href):
    """Troca o redirecionamento do Google (/url?q=...) pelo link direto da loja."""
    if not href:
        return ""
    if href.startswith("/"):
        href = "https://www.google.com" + href
    url = urlparse(href)
    parametros = parse_qs(url.query)
    if url.path == "/url" and ("q" in parametros or "url" in parametros):
        return (parametros.get("q") or parametros.get("url"))[0]
    return href


def link_de_busca(titulo):
    return URL_BUSCA.format(termo=quote_plus(titulo))


# ---------- Leitura da página ----------

JS_EXTRAIR_CARDS = r"""
() => {
  const precoRe = /R\$\s?\d[\d.]*(,\d{2})?/;
  const ruido = /^(R\$|frete|entrega|\d+([.,]\d+)?\s*(\(|$)|avalia|compar|mais lojas|loja(s)? (oficial|verificada)|usado|\+|em até|à vista|ver |patrocinad|anúncio|oferta|desconto|-?\d+%)/i;
  const visto = new Set(), out = [];
  const folhas = [...document.querySelectorAll('span,div,b,strong')]
    .filter(e => e.children.length === 0 && precoRe.test(e.textContent) && e.textContent.length < 40);
  for (const f of folhas) {
    if (getComputedStyle(f).textDecorationLine.includes('line-through')) continue; // preço antigo riscado
    let card = f.parentElement;
    for (let i = 0; i < 10 && card; i++, card = card.parentElement) {
      const linhas = card.innerText.split('\n').map(s => s.trim()).filter(Boolean);
      if (linhas.length >= 3 && (card.matches('a[href]') || card.querySelector('a[href]') || card.closest('a[href]'))) break;
    }
    if (!card || visto.has(card)) continue;
    // ignora containers grandes (lista inteira) – um card tem só 1 ou 2 preços
    const precos = card.innerText.match(new RegExp(precoRe.source, 'g')) || [];
    if (precos.length > 3) continue;
    visto.add(card);
    const linhas = card.innerText.split('\n').map(s => s.trim()).filter(Boolean);
    const iPreco = linhas.findIndex(l => l.includes(f.textContent.trim()));
    const candidatos = linhas.filter(l => !precoRe.test(l) && !ruido.test(l));
    const titulo = candidatos.slice().sort((a, b) => b.length - a.length)[0] || '';
    let loja = '';
    for (const l of linhas.slice(iPreco + 1)) {
      if (!precoRe.test(l) && !ruido.test(l) && l !== titulo && l.length <= 40) { loja = l; break; }
    }
    const links = [...card.querySelectorAll('a[href]')].map(a => a.href);
    if (card.closest('a[href]')) links.unshift(card.closest('a[href]').href);
    const externo = links.find(h => !/google\./.test(new URL(h, location.href).hostname));
    out.push({titulo, preco_txt: f.textContent.trim(), loja, link: externo || links[0] || ''});
  }
  return out;
}
"""


async def aceitar_cookies(pagina):
    for texto in BOTOES_COOKIES:
        botao = pagina.get_by_role("button", name=texto)
        if await botao.count():
            await botao.first.click()
            return


async def pagina_tem_captcha(pagina):
    html = (await pagina.content()).lower()
    return "/sorry/" in pagina.url or any(sinal in html for sinal in SINAIS_DE_CAPTCHA)


async def carregar_resultados(pagina, nome_item):
    await pagina.goto(URL_BUSCA.format(termo=quote_plus(nome_item)),
                      wait_until="domcontentloaded", timeout=TIMEOUT_CARREGAMENTO_MS)
    await aceitar_cookies(pagina)
    await pagina.wait_for_timeout(ESPERA_RENDERIZACAO_MS)
    if await pagina_tem_captcha(pagina):
        raise BloqueioDoGoogle()
    await pagina.mouse.wheel(0, 3000)  # força o carregamento de mais cards
    await pagina.wait_for_timeout(ESPERA_ROLAGEM_MS)


def cards_para_ofertas(cards, nome_item, filtrar_titulo):
    """Converte os cards brutos em Ofertas válidas, sem duplicatas."""
    ofertas, vistas = [], set()
    for card in cards:
        preco = extrair_preco(card["preco_txt"])
        if preco is None or not card["titulo"]:
            continue
        if filtrar_titulo and not titulo_corresponde_ao_item(nome_item, card["titulo"]):
            continue
        chave = (card["titulo"].lower(), card["loja"].lower(), preco)
        if chave in vistas:
            continue
        vistas.add(chave)
        ofertas.append(Oferta(titulo=card["titulo"], preco=preco, loja=card["loja"],
                              link=link_final(card["link"]) or link_de_busca(card["titulo"])))
    return ofertas


def mais_baratas(ofertas, quantidade):
    return sorted(ofertas, key=lambda o: o.preco)[:quantidade]


async def buscar_item(pagina, item, quantidade, filtrar_titulo):
    """Pesquisa um item e devolve um ResultadoBusca (nunca lança erro, exceto bloqueio)."""
    try:
        await carregar_resultados(pagina, item.nome)
        cards = await pagina.evaluate(JS_EXTRAIR_CARDS)
    except BloqueioDoGoogle:
        raise
    except Exception as erro:
        return ResultadoBusca(item, f"{Status.ERRO}: {type(erro).__name__}")

    ofertas = mais_baratas(cards_para_ofertas(cards, item.nome, filtrar_titulo), quantidade)
    return ResultadoBusca(item, Status.OK if ofertas else Status.SEM_RESULTADO, ofertas)

## 6. Orquestração

In [ ]:
import asyncio
import random

from playwright.async_api import async_playwright


async def abrir_navegador(playwright):
    navegador = await playwright.chromium.launch(
        headless=True, args=["--disable-blink-features=AutomationControlled"])
    contexto = await navegador.new_context(
        locale="pt-BR", user_agent=USER_AGENT, viewport={"width": 1366, "height": 900})
    return navegador, await contexto.new_page()


async def buscar_todos(itens):
    """Busca item a item. Se o Google bloquear, os restantes ficam PENDENTES."""
    resultados = []
    async with async_playwright() as playwright:
        navegador, pagina = await abrir_navegador(playwright)
        for posicao, item in enumerate(itens, start=1):
            try:
                resultado = await buscar_item(pagina, item, QTD_OFERTAS, FILTRAR_TITULO)
            except BloqueioDoGoogle:
                resultados.append(ResultadoBusca(item, Status.BLOQUEADO))
                resultados += [ResultadoBusca(i, Status.PENDENTE) for i in itens[posicao:]]
                print(f"⛔ Google bloqueou na busca {posicao}. Espere alguns minutos e rode de novo.")
                break
            resultados.append(resultado)
            print(f"[{posicao}/{len(itens)}] {resultado.status:<14} {item.nome[:70]}")
            await asyncio.sleep(random.uniform(*PAUSA_ENTRE_BUSCAS_SEG))
        await navegador.close()
    return resultados


def ordenar_como_na_origem(linhas, itens):
    ordem = {item.item_id: i for i, item in enumerate(itens)}
    return sorted(linhas, key=lambda l: (ordem.get(l[0], len(ordem)), str(l[4])))


async def executar(planilha):
    itens = ler_itens_selecionados(planilha)
    aba_destino = obter_aba_destino(planilha)

    ids_selecionados = {item.item_id for item in itens}
    linhas_mantidas = [l for l in ler_linhas_concluidas(aba_destino) if l[0] in ids_selecionados] \
        if PULAR_JA_CONCLUIDOS else []
    ids_concluidos = {l[0] for l in linhas_mantidas}
    pendentes = [item for item in itens if item.item_id not in ids_concluidos]

    print(f"{len(itens)} itens selecionados | {len(ids_concluidos)} já concluídos | "
          f"{len(pendentes)} para buscar\n")
    resultados = await buscar_todos(pendentes)

    linhas_novas = [linha for r in resultados for linha in resultado_para_linhas(r)]
    gravar_resultados(aba_destino, ordenar_como_na_origem(linhas_mantidas + linhas_novas, itens))
    print(f"\n✅ Resultado gravado na aba '{ABA_DESTINO}'.")

## ▶️ 7. Executar

In [ ]:
from google.colab import auth
from google.auth import default

auth.authenticate_user()
credenciais, _ = default()
planilha = abrir_planilha(gspread.authorize(credenciais), PLANILHA_URL)

await executar(planilha)

### Status possíveis
| Status | Significado |
|---|---|
| `OK` | Ofertas encontradas |
| `SEM RESULTADO` | Nenhuma oferta (ou nenhuma passou no filtro de título). Teste `FILTRAR_TITULO = False` |
| `BLOQUEADO (captcha)` | O Google bloqueou nesta busca |
| `PENDENTE` | Não foi buscado por causa do bloqueio. Rode de novo mais tarde |
| `ERRO: ...` | Falha inesperada ao carregar a página |